# Patient Experience — Detractor / Grievance Risk Model (XGBoost)

Headless-executable training notebook (Phase 3 of the Patient Experience Command Center accelerator).
Trains a custom **XGBoost** classifier on `FS_PX_RISK_FEATURES`, registers it to the **Model Registry**
with SQL scoring + SHAP explainability enabled, and writes a model-derived global **driver-importance**
table. Downstream `MDL_DISSAT_RISK` (rendered SQL) scores every encounter in-database via
`MODEL!PREDICT_PROBA` + `MODEL!EXPLAIN`.

In [ ]:
import sys
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Headless NPOs have NO implicit database/schema, so never rely on get_current_database().
# Target db/schema arrive as ARGUMENTS (config-driven, portable); fall back to current context if set.
if len(sys.argv) >= 3:
    DB, SCH = sys.argv[1], sys.argv[2]
else:
    DB  = (session.get_current_database()  or 'CUSTOMER_DERIVED_DB').strip('"')
    SCH = (session.get_current_schema()    or 'PATIENT_EXPERIENCE').strip('"')
session.sql(f'USE DATABASE {DB}').collect()
session.sql(f'USE SCHEMA {SCH}').collect()

FEATURES   = f"{DB}.{SCH}.FS_PX_RISK_FEATURES"
MODEL_NAME = "PX_DISSAT_RISK_XGB"
VERSION    = "V1"

NUM = ["F_MED_REC","F_INTERPRETER_USED","F_INTERPRETER_NEEDED","F_CALLBACK_REACHED",
       "F_CALLBACK_ISSUE","F_PRIOR_GRIEVANCE_CNT","F_COMMENT_SENTIMENT",
       "F_HAS_NEGATIVE_COMMENT","F_COMMENT_COUNT"]
CAT = ["F_UNIT_ID","F_RACE","F_ETHNICITY","F_LANGUAGE","PERIOD_BUCKET"]
FEATS = NUM + CAT
print("target:", FEATURES)

In [ ]:
sdf = (session.table(FEATURES)
       .filter("LABEL_DETRACTOR IS NOT NULL")
       .select(*FEATS, "LABEL_DETRACTOR"))
pdf = sdf.to_pandas()
print("labeled rows:", len(pdf))
print(pdf["LABEL_DETRACTOR"].value_counts().to_dict())

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

for c in NUM:
    pdf[c] = pdf[c].astype(float).fillna(0.0)
for c in CAT:
    pdf[c] = pdf[c].astype(str).fillna("UNK")

X = pdf[FEATS]
y = pdf["LABEL_DETRACTOR"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), CAT)],
    remainder="passthrough")
clf = Pipeline([
    ("pre", pre),
    ("xgb", XGBClassifier(n_estimators=250, max_depth=5, learning_rate=0.08,
                           subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                           eval_metric="logloss", random_state=42))])
clf.fit(Xtr, ytr)

p = clf.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, p); ap = average_precision_score(yte, p)
print(f"holdout ROC-AUC={auc:.4f}  PR-AUC={ap:.4f}")

In [ ]:
from snowflake.ml.registry import Registry
reg = Registry(session=session, database_name=DB, schema_name=SCH)

# Clean slate so the notebook is re-runnable headless (can't delete a default version in place).
session.sql(f"DROP MODEL IF EXISTS {DB}.{SCH}.{MODEL_NAME}").collect()
print("dropped prior model if any")

log_kwargs = dict(
    model=clf, model_name=MODEL_NAME, version_name=VERSION,
    sample_input_data=Xtr.head(200),
    comment="PX detractor/grievance risk - XGBoost pipeline (OneHot + XGBClassifier)",
    target_platforms=["WAREHOUSE"],
    options={"enable_explainability": True})
try:
    mv = reg.log_model(**log_kwargs)
    EXPLAIN_OK = True
except Exception as e:
    print("explainability log failed, retrying without:", type(e).__name__, str(e)[:200])
    log_kwargs["options"] = {}
    mv = reg.log_model(**log_kwargs)
    EXPLAIN_OK = False

reg.get_model(MODEL_NAME).default = VERSION
fns = [f["name"] if isinstance(f, dict) else f.name for f in mv.show_functions()]
print("registered", MODEL_NAME, VERSION, "| explain_enabled=", EXPLAIN_OK)
print("functions:", fns)

In [ ]:
import pandas as pd
xgb = clf.named_steps["xgb"]
ohe = clf.named_steps["pre"].named_transformers_["cat"]
enc_names = list(ohe.get_feature_names_out(CAT)) + NUM

def base_feature(f):
    for c in CAT:
        if f.startswith(c + "_"):
            return c
    return f

imp = pd.DataFrame({"ENC_FEATURE": enc_names, "IMPORTANCE": xgb.feature_importances_})
imp["DRIVER"] = imp["ENC_FEATURE"].apply(base_feature)
agg = (imp.groupby("DRIVER", as_index=False)["IMPORTANCE"].sum()
          .sort_values("IMPORTANCE", ascending=False).reset_index(drop=True))
agg["IMPORTANCE"] = (agg["IMPORTANCE"] * 100).round(2)
agg["RANK"] = agg.index + 1
agg = agg[["RANK", "DRIVER", "IMPORTANCE"]]

session.write_pandas(agg, "MDL_DRIVER_IMPORTANCE", database=DB, schema=SCH,
                     auto_create_table=True, overwrite=True, quote_identifiers=False)
print(agg.to_string(index=False))

Model `PX_DISSAT_RISK_XGB` (V1, default) is registered for WAREHOUSE scoring and
`MDL_DRIVER_IMPORTANCE` is written. Next (rendered SQL): `MDL_DISSAT_RISK` scores all encounters
with `MODEL!PREDICT_PROBA` + per-row `MODEL!EXPLAIN` SHAP reasons.